# RPS-1 Project: Shared Specification

All three notebooks (RPS_detection, RPS_strategy, RPS_evaluation) must follow this file. Changes are agreed in the team chat first, then recorded here with a new version number.

Spec version: 1.0

## 1. Labels

In [ ]:
SHAPES = ['Rock', 'Paper', 'Scissors']
PLAYERS = ['self', 'opponent']
NUM_HANDS = 4
SEED = 42
PAYOFF = {'win': 1, 'tie': 0, 'loss': -1}
SPEC_VERSION = '1.0'

## 2. Hand record format

Each detected hand is a dictionary with these keys:
- player: 'self' (upright hands) or 'opponent' (upside-down hands)
- shape: one of SHAPES
- position: 0 to 3, counted left to right in the image
- confidence: float from 0 to 1, or None

Detection returns exactly four records, sorted by position. Each player has two.

In [ ]:
EXAMPLE_DETECTION = [
    {'player': 'self', 'shape': 'Paper', 'position': 0, 'confidence': 0.95},
    {'player': 'self', 'shape': 'Scissors', 'position': 1, 'confidence': 0.90},
    {'player': 'opponent', 'shape': 'Paper', 'position': 2, 'confidence': 0.88},
    {'player': 'opponent', 'shape': 'Rock', 'position': 3, 'confidence': 0.93},
]

## 3. Function contracts

Each notebook implements its own function. The other notebooks call it with the formats above. Do not change a signature without updating this spec.

In [ ]:
def check_hands(hands):
    '''Validate a list of hand records against Section 2. Raises ValueError on any violation. Returns True otherwise.'''
    if len(hands) != NUM_HANDS:
        raise ValueError('expected 4 hands, got ' + str(len(hands)))
    if sorted(h['position'] for h in hands) != [0, 1, 2, 3]:
        raise ValueError('positions must be 0 to 3, each used once')
    for h in hands:
        if h['player'] not in PLAYERS:
            raise ValueError('bad player: ' + str(h['player']))
        if h['shape'] not in SHAPES:
            raise ValueError('bad shape: ' + str(h['shape']))
    counts = {p: sum(1 for h in hands if h['player'] == p) for p in PLAYERS}
    if counts['self'] != 2 or counts['opponent'] != 2:
        raise ValueError('need 2 hands per player, got ' + str(counts))
    return True


def detect_hands(image_path):
    '''Owned by RPS_detection. Return a list of 4 hand records (Section 2). Raise ValueError if the number of hands is not 4 or a shape cannot be classified. Never return a partial list.'''
    raise NotImplementedError('Owned by RPS_detection')


def decide(self_hands, opponent_hands, rng=None):
    '''Owned by RPS_strategy. Return a dict with keys: keep_position (0 or 1, index into self_hands), keep_shape (a value from SHAPES), probabilities (dict with keys 0 and 1, values summing to 1), sampled (True when keep_position was drawn at random).'''
    raise NotImplementedError('Owned by RPS_strategy')

## 4. Files and paths

All paths are relative to the repository root.
- data/demo/: five demo images, named demo_01.jpg to demo_05.jpg
- data/test/: separate test set, never used for training or tuning
- data/expected_labels.json: maps each image name to its four expected hand records
- models/: model weights, or a note in RPS_detection on where to download them

## 5. Packages

Pin versions after the first working install, then record them with pip freeze in RPS_evaluation.

In [ ]:
# !pip install numpy matplotlib opencv-python

## 6. Worked check for the strategy

From the project notes: Player-1 shows Paper and Scissors, and Player-2 shows Paper and Rock. Player-1 keeps the Paper hand with probability 2/3 and the Scissors hand with probability 1/3. The expected payoff is 1/3. RPS_strategy must reproduce these numbers.

In [ ]:
EXPECTED_EXAMPLE = {
    'self_shapes': ['Paper', 'Scissors'],
    'opponent_shapes': ['Paper', 'Rock'],
    'probability_keep_position_0': 2 / 3,
    'probability_keep_position_1': 1 / 3,
    'expected_payoff': 1 / 3,
}